In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

import boto3
import os

session = boto3.Session(
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

In [ ]:
import mlflow

aws_path = os.getenv("AWS_INSTANCE_IP")
mlflow.set_tracking_uri(f"http://{aws_path}:5000")

In [3]:
mlflow.set_experiment('Exp 6 - LightGBM HP Tuning')

<Experiment: artifact_location='s3://mlflow-bucket-carlos/6', creation_time=1772600117080, experiment_id='6', last_update_time=1772600117080, lifecycle_stage='active', name='Exp 6 - LightGBM HP Tuning', tags={}>

In [4]:
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
df = pd.read_csv('reddit_preprocessing.csv')
df.dropna(inplace = True)
print(df.shape)

df.head()

(36662, 2)


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [6]:
df.category = df.category.map({-1:2, 0:0, 1:1})

df.dropna(subset = ['category'], inplace = True)

In [7]:
ngram_range = (1,3)
max_features = 1000

vectorizer = TfidfVectorizer(ngram_range = ngram_range, max_features = max_features)

X = vectorizer.fit_transform(df.clean_comment)
y = df.category

smote = SMOTE(random_state = 42)

X_resampled, y_resampled = smote.fit_resample(X,y)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size = 0.2, random_state = 42, stratify = y_resampled)

In [9]:
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    with mlflow.start_run():
        # Log model type and trial number
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")

        return accuracy

In [10]:
def objective_lightgbm(trial):
    # Hyperparameter space to explore
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)  # L1 regularization
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)  # L2 regularization

    # Log trial parameters
    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda
    }

    # Create LightGBM model
    model = LGBMClassifier(n_estimators=n_estimators,
                           learning_rate=learning_rate,
                           max_depth=max_depth,
                           num_leaves=num_leaves,
                           min_child_samples=min_child_samples,
                           colsample_bytree=colsample_bytree,
                           subsample=subsample,
                           reg_alpha=reg_alpha,
                           reg_lambda=reg_lambda,
                           random_state=42,
                           verbosity = -1,
                           verbose = -1)

    # Log each trial as a separate run in MLflow
    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy


In [11]:
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=5)  # Increased to 100 trials

    # Get the best parameters
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'],
                                learning_rate=best_params['learning_rate'],
                                max_depth=best_params['max_depth'],
                                num_leaves=best_params['num_leaves'],
                                min_child_samples=best_params['min_child_samples'],
                                colsample_bytree=best_params['colsample_bytree'],
                                subsample=best_params['subsample'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'],
                                random_state=42,
                                verbosity = -1,
                                verbose = -1)

    # Log the best model with MLflow and print the classification report
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best")

    # Plot parameter importance
    optuna.visualization.plot_param_importances(study).show()

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study).show()

In [12]:
# Run the experiment for LightGBM
run_optuna_experiment()

[I 2026-03-04 02:53:19,665] A new study created in memory with name: no-name-e455c1df-3648-4622-bb83-0cf6f434f1da
/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/04 02:54:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Trial_0_LightGBM_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6/runs/b0e20a882f8f4bf7b094aa4195ef49c8
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6


[I 2026-03-04 02:54:06,851] Trial 0 finished with value: 0.8014161910801099 and parameters: {'n_estimators': 913, 'learning_rate': 0.014016186654283098, 'max_depth': 10, 'num_leaves': 150, 'min_child_samples': 56, 'colsample_bytree': 0.5377570424413856, 'subsample': 0.7584266460773071, 'reg_alpha': 0.524632443471693, 'reg_lambda': 1.6488393637974283}. Best is trial 0 with value: 0.8014161910801099.
/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/04 02:54:31 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Trial_1_LightGBM_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6/runs/2007a113eab541798d13b46265d46eeb
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6


[I 2026-03-04 02:54:36,973] Trial 1 finished with value: 0.6085394208412598 and parameters: {'n_estimators': 312, 'learning_rate': 0.00030171644967494035, 'max_depth': 7, 'num_leaves': 49, 'min_child_samples': 22, 'colsample_bytree': 0.9414684204598647, 'subsample': 0.9563040171137817, 'reg_alpha': 0.011948116445262828, 'reg_lambda': 0.2704031253228432}. Best is trial 0 with value: 0.8014161910801099.
/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/04 02:55:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run Trial_3_LightGBM_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6/runs/a9afebeb3af7461193d4a0e16f03007d
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6


[I 2026-03-04 02:56:01,058] Trial 3 finished with value: 0.8046924540266328 and parameters: {'n_estimators': 829, 'learning_rate': 0.04003265331883936, 'max_depth': 5, 'num_leaves': 77, 'min_child_samples': 30, 'colsample_bytree': 0.5804375606097457, 'subsample': 0.9626173898901297, 'reg_alpha': 0.0008340497600116899, 'reg_lambda': 0.0005625874325102861}. Best is trial 2 with value: 0.8101881209046713.
/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/04 02:56:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Trial_4_LightGBM_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6/runs/12b4918813a74ce1a2639d2b6f11383d
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6


[I 2026-03-04 02:56:31,249] Trial 4 finished with value: 0.6070598182202495 and parameters: {'n_estimators': 781, 'learning_rate': 0.00012129827332392713, 'max_depth': 4, 'num_leaves': 95, 'min_child_samples': 46, 'colsample_bytree': 0.5838475553588565, 'subsample': 0.6577358140308847, 'reg_alpha': 0.03910132458513434, 'reg_lambda': 0.0014407998835843782}. Best is trial 2 with value: 0.8101881209046713.
/opt/homebrew/Caskroom/miniforge/base/envs/ai/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/03/04 02:57:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Trial_Best_LightGBM_SMOTE_TFIDF_Trigrams at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6/runs/ba0cdca16be146698eef7e963ef4305d
🧪 View experiment at: http://ec2-98-89-26-222.compute-1.amazonaws.com:5000/#/experiments/6


In [13]:
best_model

NameError: name 'best_model' is not defined